# **1) Project Title**
*   Customer Churn Prediction.


# **2) Project Overview**
*  This project aims to build a Machine Learning model that predicts whether a telecom customer will churn (leave the service) based on demographic information, account details, and service usage.
*   The system analyzes customer behavior and identifies patterns that indicate potential churn, helping companies improve customer retention strategies.

In [1]:
# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

# **3) Problem Statement**
*   Customer churn is a major challenge for telecom companies. Losing customers leads to revenue loss and increased marketing costs for acquiring new customers.
*   The goal of this project is to:
    *   Predict whether a customer will churn.
    *   Identify the most important factors influencing churn.
    *   Provide insights that help reduce churn rates.

# **4) Dataset Description**
*  The dataset contains information about telecom customers including:
    *   Customer demographics.
    *   Account information.
    *   Services subscribed.
    *   Payment methods.
    *   Monthly and total charges.

In [ ]:
# 2. Load Dataset

data = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Dataset Shape:", data.shape)
data.head()

# **5) Data Exploration**
*   Exploratory Data Analysis (EDA) helps us understand the dataset distribution and relationships between variables.



In [ ]:
# 3. Data Exploration

In [ ]:
# 3.1. Dataset Information

print(data.info())

In [ ]:
# 3.2. Statistical Summary

print(data.describe())

In [ ]:
# 3.3. Check Missing Values

print(data.isnull().sum())

# **6) Data Preprocessing**
*   Data preprocessing prepares the dataset for machine learning.
*   Steps include:
    *   Handling missing values.
    *   Converting data types.
    *   Encoding categorical variables.
    *   Scaling features.

In [6]:
# 4. Data Cleaning

# Convert TotalCharges to numeric
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Fill missing values
data["TotalCharges"].fillna(data["TotalCharges"].median(), inplace=True)

# Remove customerID
data.drop("customerID", axis=1, inplace=True)

In [7]:
# 5. Data Visualization

In [ ]:
# 5.1. Churn Distribution Visualization

plt.figure(figsize=(6,4))
sns.countplot(x="Churn", data=data)
plt.title("Churn Distribution")
plt.show()

In [ ]:
# 5.2. Contract vs Churn Visualization

plt.figure(figsize=(8,5))
sns.countplot(x="Contract", hue="Churn", data=data)
plt.title("Contract Type vs Churn")
plt.show()

In [ ]:
# 5.3. Monthly Charges Distribution

plt.figure(figsize=(6,4))
sns.histplot(data["MonthlyCharges"], bins=30, kde=True)
plt.title("Monthly Charges Distribution")
plt.show()

In [11]:
# 6. Encode Categorical Variables

encoder = LabelEncoder()
for column in data.columns:
    if data[column].dtype == "object":
        data[column] = encoder.fit_transform(data[column])

In [ ]:
# 7. Correlation Heatmap

plt.figure(figsize=(14,10))
sns.heatmap(data.corr(), cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

In [13]:
# 8. Split Features and Target

X = data.drop("Churn", axis=1)
y = data["Churn"]

In [14]:
# 9. Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
# 10. Feature Scaling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# **7) Model Development**
*   We train multiple machine learning models and compare their performance.
*   Models used:
    *   Logistic Regression.
    *   Decision Tree.
    *   Random Forest.
    *   Support Vector Machine.
    *   XGBoost.

In [ ]:
# 11. Train Multiple Models

models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=200),
    "Support Vector Machine": SVC(),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy
    print("\n",name)
    print("Accuracy:",accuracy)

In [ ]:
# 12. Model Comparison

plt.figure(figsize=(8,5))
plt.bar(results.keys(), results.values())
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.show()

In [18]:
# 13. Best Model (XGBoost)

xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss")
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

# **8) Model Evaluation**
*   We evaluate models using:
    *   Accuracy.
    *   Confusion Matrix.
    *   Classification Report.

In [ ]:
# 14. Model Evaluation

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("\nConfusion Matrix")
cm = confusion_matrix(y_test, xgb_pred)
print(cm)
print("\nClassification Report")
print(classification_report(y_test, xgb_pred))

# **9) Visualization**
*   Visualization helps understand patterns and model results.

In [ ]:
# 15. Confusion Matrix Visualization

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 16. Feature Importance

importances = xgb_model.feature_importances_
features = X.columns
importance_df = pd.DataFrame({"Feature": features, "Importance": importances}).sort_values(by="Importance", ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x="Importance", y="Feature", data=importance_df)
plt.title("Feature Importance")
plt.show()

In [ ]:
# 17. Hyperparameter Tuning (GridSearch)

param_grid = {"n_estimators": [200,300], "max_depth": [4,6,8], "learning_rate": [0.01,0.05,0.1]}
grid = GridSearchCV(XGBClassifier(eval_metric="logloss"), param_grid, cv=3, scoring="accuracy")
grid.fit(X_train, y_train)
print("Best Parameters:", grid.best_params_)

In [ ]:
# 18. Final Model

final_model = grid.best_estimator_
predictions = final_model.predict(X_test)
print("Final Accuracy:", accuracy_score(y_test, predictions))

In [ ]:
# 19. Save Model

import joblib
joblib.dump(final_model, "customer_churn_model.pkl")
print("Model saved successfully")

# **10) Tools & Technologies**
*   This project uses the following technologies:
    *   Python.
    *   Pandas.
    *   NumPy.
    *   Scikit-learn.
    *   XGBoost.
    *   Matplotlib.
    *   NumPy.
    *   Seaborn.
    *   Google Colab.

# **11) Deliverables**
*   The final deliverables include:
    *   Complete Machine Learning code.
    *   Data preprocessing pipeline.
    *   Model training and evaluation.
    *   Visualization charts.
    *   Saved prediction model.